# LangGraph End-to-End Workflow — Repository Intelligence

## 1 — Learning objectives and architecture

This notebook demonstrates the **complete multi-agent repository-query
workflow running on LangGraph**, from the initial question through
planning, task execution, evidence commit, final synthesis, and
deterministic validation — using the repository's *real* implementation.

**What you will learn**

- how the production entry point (`run_repository_query`) selects
  between the legacy M17 orchestrator loop and the M21 LangGraph task
  loop (`orchestration="langgraph"`);
- what the **Planner**, **Investigators**, and the **Analyst** actually
  do (and which contracts tie them together);
- how a compiled `StateGraph` owns task selection, execution, result
  commit, retry, and final assembly — while the planner stays
  *off-graph*;
- how evidence and provenance flow from investigator results into the
  final answer;
- how failures (invalid plans, task failures, exhausted retries)
  terminate honestly;
- how the H1–H4 request-budget and graph-projection safeguards apply
  to the graph path unchanged.

**The architecture (as implemented — not aspirational)**

```mermaid
flowchart TD
    Q[Repository Query] --> P[Planner<br/>build_planning_request → LLM → plan_from_response]
    P --> PG[H1 request-budget preflight]
    PG --> VP[validate_plan]
    VP --> RA[assign_roles<br/>INVESTIGATOR… ANALYST]
    RA --> G[LangGraph task loop<br/>initialize → select_task → execute_task → commit_result<br/>⇄ retry_task ; terminal → assemble_and_validate]
    G --> AR[run_with_langgraph returns OrchestrationRun]
    AR --> ASM[_assemble: answer, six-rule diagram validation,<br/>artifacts → RepositoryQueryResult]
    ASM --> R[Repository Query Result]
```

Key placement decisions verified against source:

- the **planner runs off-graph** (both backends call the same
  `_build_plan`, with the H1 request-budget preflight);
- the **graph owns the task loop only** — task selection, attempts,
  retry, commits, blocking, and the M16 collaboration validation
  (`assemble_and_validate`);
- the **legacy path stays the default**; the graph path is opt-in and
  is never silently substituted after a failure.


## 2 — Environment and setup

The repository ships a deterministic **offline mode** (scripted models
and canned tool bindings) that is the default for this notebook — no
network and **no credentials are required** for any cell below.

Requirements:

- the project virtual environment `venv312` (Python 3.12) with the
  package installed editable (`pip install -e .`);
- the optional `langgraph` dependency (pinned `langgraph==1.2.15`,
  declared as the `graph` extra) — used by the M21 task loop.

Credentials (only for the *optional* live demonstration, not run
here): set `OPENAI_API_KEY` in your environment and call
`run_repository_query(..., mode="live")`. Never paste keys into
notebook cells. Live mode fails loudly when unconfigured — it never
falls back to the offline model.


In [ ]:
# -- locate the repository root WITHOUT hardcoding a machine path -------
import sys
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    """Walk up from the notebook's working directory until the
    project marker (pyproject.toml + src/repo_intelligence) is found.
    Works whether the kernel starts in the repo root or in notebooks/."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "repo_intelligence"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "run this notebook from inside the repository checkout"
    )

ROOT = find_repo_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# -- dependency + mode checks -------------------------------------------
import platform

import repo_intelligence
from repo_intelligence import repository_query as rq

try:
    import langgraph.version as _lg_version
    GRAPH_AVAILABLE = True
    GRAPH_REASON = _lg_version.__version__
except ImportError as exc:  # optional extra is genuinely optional
    GRAPH_AVAILABLE = False
    GRAPH_REASON = str(exc)

print("python           :", platform.python_version())
print("repo root        :", ROOT.name)
print(
    "repo_intelligence:",
    "editable checkout" if str(ROOT) in (repo_intelligence.__file__ or "")
    else repo_intelligence.__file__,
)
print("langgraph        :", GRAPH_REASON if GRAPH_AVAILABLE else "MISSING — " + GRAPH_REASON)
print("offline default  : deterministic, no credentials needed")


Every graph-dependent cell below calls `_require_graph()` first, so a
missing optional dependency fails loudly with instructions instead of
silently skipping work — the same no-silent-fallback discipline the
production entry point uses.


In [ ]:
def _require_graph():
    if not GRAPH_AVAILABLE:
        raise RuntimeError(
            "this cell needs the optional 'langgraph' dependency "
            "(pip install 'repo_intelligence[graph]' or "
            "pip install langgraph==1.2.15)"
        )
    return True

print("graph cells ready:", GRAPH_AVAILABLE)


## 3 — Inspect the real components

The notebook imports the production modules; it never re-implements
them. Everything below is a *view* onto the source you can open
side-by-side:

| Component | Source |
| --- | --- |
| entry point | `src/repo_intelligence/repository_query.py` |
| planner request/validation | `src/repo_intelligence/planning/` |
| task contracts | `src/repo_intelligence/orchestration/contracts.py` |
| role vocabulary | `src/repo_intelligence/orchestration/roles.py` |
| legacy loop | `src/repo_intelligence/orchestration/runner.py` |
| LangGraph task loop | `src/repo_intelligence/orchestration/langgraph_runner.py` |


In [ ]:
# -- the public entry point and the orchestration seam -------------------
import inspect

print("run_repository_query" + str(inspect.signature(rq.run_repository_query)))
print()
print("orchestration backends:", rq.ORCHESTRATION_MODES,
      "(default:", rq.ORCHESTRATION_LEGACY + ")")
print()
print("--- _run_with_langgraph (the opt-in seam, source verbatim) ---")
print(inspect.getsource(rq._run_with_langgraph))


In [ ]:
# -- the planner: OFF-graph, budget-gated, validated ---------------------
print("--- _build_plan (planner path used by BOTH backends) ---")
print(inspect.getsource(rq._build_plan))


In [ ]:
# -- the typed contracts that tie the roles together ---------------------
import dataclasses

from repo_intelligence.multiagent import TaskPayload
from repo_intelligence.orchestration import (
    ANALYST,
    INVESTIGATOR,
    RoleAssignment,
    assign_roles,
)
from repo_intelligence.planning.models import InvestigationPlan, PlanStep

def fields_of(model):
    """Field names of a pydantic v2 model or a dataclass."""
    if dataclasses.is_dataclass(model):
        return [f.name for f in dataclasses.fields(model)]
    return list(model.model_fields)

print("AGENT_ROLES            :", (INVESTIGATOR, ANALYST))
print("InvestigationPlan      :", fields_of(InvestigationPlan))
print("PlanStep               :", fields_of(PlanStep))
print("RoleAssignment         :", fields_of(RoleAssignment))
print("TaskPayload            :", fields_of(TaskPayload))
print("RepositoryQueryResult  :", fields_of(rq.RepositoryQueryResult))


## 4 — Build (compile) the actual graph

The production wiring lives in
`orchestration/langgraph_runner.py`. One compiled `StateGraph` owns the
whole task loop; the **plan is data, not topology** (a 2-step or a
4-step plan run on the same compiled graph). Below we compile the real
graph and print its nodes and edges — no graph is re-built in this
notebook.


In [ ]:
_require_graph()
from repo_intelligence.orchestration import langgraph_runner as lg

# -- the graph state the nodes read and write ---------------------------
print("LangGraphOrchestrationState keys:")
for key in sorted(lg.LangGraphOrchestrationState.__annotations__):
    print("  -", key)


In [ ]:
# -- compile the REAL graph and inspect its topology ---------------------
# (a runtime is only needed when INVOKING nodes; compile-time we pass
# None — node closures capture the runtime parameter slot)
compiled = lg.build_langgraph_orchestration_graph(None)
topology = compiled.get_graph()

print("nodes:")
for name in topology.nodes:
    print("  ", name)
print("edges:")
for edge in topology.edges:
    kind = "conditional" if edge.conditional else "hard"
    label = ("  [%s]" % edge.data) if edge.data else ""
    print("   %-22s -> %-22s %s%s" % (edge.source, edge.target, kind, label))


Reading the topology:

- `initialize` — fresh `OrchestrationRun`, `validate_plan`, role
  validation (`_preassigned_role_assignments`), chain dependencies;
- `select_task` — next pending step by index; budget/timeout pre-task
  gate; **mid-run merge of investigator reports before the ANALYST
  step** (the analyst consumes the assembled report through
  `dependency_results`);
- `execute_task` — the single `executor.execute(request)` dispatch
  (M19 `AgentTaskRequest` in, `AgentTaskResult` out, then
  `validate_task_result`);
- `commit_result` — honest commit or typed failure; retry decision
  *inside* the retry branch so a run-level budget stop never consumes
  a retry (M20 rule);
- `retry_task` — same request identity, attempt+1, same trace window;
- `assemble_and_validate` — `RunAssembler.assemble` + the M16
  deterministic `validate_collaboration` verdict → terminal status.

The production helper `run_with_langgraph(runtime, task, plan,
role_assignments)` compiles this topology and invokes it, returning the
authoritative `OrchestrationRun`.


In [ ]:
print(inspect.signature(lg.run_with_langgraph))


## 5 — Execute an end-to-end query (production entry point)

One question through the **actual production entry point** with
`orchestration="langgraph"` — offline, deterministic, no credentials.
Everything the orchestrator needs (subject resolution, snapshot
verification, planner + H1 preflight, specialists, budgets, assembly,
six-rule diagram validation) is the production composition.


In [ ]:
_require_graph()
import tempfile

QUESTION = "How do the agents, tools, and MCP servers interact?"
WORK = Path(tempfile.mkdtemp(prefix="nb_query_"))

result = rq.run_repository_query(
    QUESTION,
    mode="offline",                      # deterministic scripted models
    diagram=True,                        # six-rule validation in scope
    artifacts=False,                     # write nothing in this notebook
    db_path=str(WORK / "memory.db"),
    trace_root=str(WORK / "traces"),
    orchestration="langgraph",           # <-- the opt-in graph path
)

print("status              :", result.status)
print("termination         :", result.termination_reason)
print("collaboration (M16) :", result.collaboration_status)
print("diagram (6 rules)   :", result.diagram_status)
print()
print("steps:")
for step in result.steps:
    print("  %-4s %-12s %-10s label=%-13s run=%s" % (
        step.step_id, step.role, step.status, step.label,
        (step.run_id or "")[:8],
    ))
print()
print("evidence lines      :", len(result.evidence))
print("conclusions         :", len(result.conclusions))
print("uncertainties       :", len(result.uncertainties))
print("answer (first line) :", result.answer.splitlines()[0][:72])


The same run from the CLI (offline default):

```bash
venv312/bin/python agentrun_repository_query.py \
  "How do the agents, tools, and MCP servers interact?" \
  --orchestration langgraph
```

Legacy remains the default: omit the flag (or pass
`--orchestration legacy`) for the M17 loop. Exit codes and the printed
result sections are identical for both backends.


## 6 — Inspect state transitions and evidence

`run_repository_query` returns the assembled public result. To see the
**run record the graph produced**, the next cell re-creates the exact
production composition — importing every piece from the source module
(`build_offline_models`, `build_architect`, `QueryRouter`,
`Orchestrator`, `_build_plan`, `assign_roles`,
`run_with_langgraph`) — so we can inspect the authoritative
`OrchestrationRun`: task records, the event timeline, the merged
investigation report, and trace attribution.


In [ ]:
_require_graph()
from repo_intelligence.orchestration import Orchestrator

LABEL = "notebook-composition"
COMMIT = rq.SUBJECT_COMMIT
TARGET = "README.md"

def compose():
    """The production composition root, assembled from the same
    builders run_repository_query uses — every piece imported from
    source, nothing re-implemented.  Fresh scripted models per call."""
    models = rq.build_offline_models(QUESTION, LABEL, COMMIT, TARGET)
    sink = rq.JsonFileTraceSink(WORK / "composition-traces")
    manager = rq.build_memory_manager(str(WORK / "composition-memory.db"))
    memory_context = rq.build_memory_context(LABEL, COMMIT)
    engine = rq.build_engine(LABEL)
    subject_root = rq.default_subject_repository()
    bindings = rq.fixture_bindings(subject_root)  # offline canned tools

    architect = rq.build_architect(
        models["arch"], sink, manager, memory_context, engine,
        bindings=bindings,
    )
    flow_investigator = rq.build_flow_investigator(
        models["flow"], sink, manager, memory_context, engine,
        bindings=bindings,
    )
    visualizer = rq.build_visualizer(
        models["viz"], sink, manager, memory_context, engine,
    )
    orchestrator = Orchestrator(
        rq.QueryRouter(architect, flow_investigator),
        visualizer,
        trace_sink=sink,
        config=rq.M17_ORCHESTRATION_CONFIG,
    )
    # planner stays OFF-graph; roles are preassigned with the same
    # deterministic assigner the legacy loop uses internally
    plan = rq._build_plan(models["planner"], QUESTION)
    task = TaskPayload(
        question=QUESTION, repository_scope=LABEL, commit_scope=COMMIT
    )
    roles = assign_roles(plan.steps)
    return orchestrator, task, plan, roles

orchestrator, task, plan, roles = compose()
print("plan steps :", [(s.id, s.goal[:38]) for s in plan.steps])
print("roles      :", [(r.step_id, r.role) for r in roles])


In [ ]:
run = lg.run_with_langgraph(orchestrator, task, plan, roles)

print("run_id     :", run.run_id[:12], "| status:", run.status,
      "| termination:", run.termination_reason)
print()
print("task records:")
for record in run.tasks:
    print("  %-4s %-12s %-10s attempts=%d deps=%s" % (
        record.task_id, record.role, record.status,
        record.attempts, record.depends_on,
    ))
print()
print("event timeline (operations, in order):")
seen = []
for event in run.events:
    op = event.get("operation")
    if op not in seen:
        seen.append(op)
for op in seen:
    print("  ", op)


In [ ]:
# -- evidence and provenance ---------------------------------------------
report = run.report
finding = run.finding
print("investigation_id :", report.investigation_id if report else None)
print("report findings  :", len(report.findings) if report else 0)
print("analyst finding  :", type(finding).__name__ if finding else None)
print("M16 verdict      :", run.validation.status if run.validation else None)
print()
if report is not None:
    print("first findings and their evidence refs:")
    for item in report.findings[:2]:
        print("  -", item.statement[:70])
        for ref in item.evidence_refs[:2]:
            print("      evidence:", ref if isinstance(ref, str) else ref)
print()
print("trace attribution (agent_runs):")
for entry in run.agent_runs:
    print("  step=%s run_id=%s" % (
        entry.get("step_id"), (entry.get("run_id") or "")[:8]
    ))


## 7 — Failure behavior

Three real failure scenarios, all deterministic — no fabricated output:

1. **Invalid plan** — `validate_plan` rejects it inside the graph's
   `initialize` node; the run terminates `PLAN_FAILED` before any
   agent executes.
2. **Plan without an Analyst** — execution succeeds for the
   investigator step, but the terminal status is
   `INCOMPLETE / NO_ANALYSIS`: a missing final synthesis is never
   fabricated into success.
3. **Retryable task failure** — a scripted investigator returns a
   broken structured response on attempt 1 (retryable by the existing
   M19/M20 policy); the graph retries within `max_retries` and
   commits attempt 2 (test-double territory — the same doubles the
   orchestration test-suite uses, loaded from
   `tests/orchestration/m17_doubles.py`).


In [ ]:
_require_graph()
from repo_intelligence.orchestration import PLAN_FAILED  # noqa: F401

# -- 1: invalid plan (no steps) ------------------------------------------
bad_plan = InvestigationPlan(objective=QUESTION, steps=[])
bad_run = lg.run_with_langgraph(orchestrator, task, bad_plan, [])

print("invalid plan  -> status:", bad_run.status,
      "| termination:", bad_run.termination_reason)
print("tasks executed:", [r.task_id for r in bad_run.tasks] or "(none)")


In [ ]:
# -- 2: plan without an Analyst (fresh composition — scripted models
#       are queued per run, so every scenario gets its own wiring) ------
solo_orch, solo_task, _p, _r = compose()
solo_plan = InvestigationPlan(
    objective=QUESTION,
    steps=[PlanStep(id="P1", goal="Identify the architecture components")],
)
solo_roles = assign_roles(solo_plan.steps)
solo_run = lg.run_with_langgraph(
    solo_orch, solo_task, solo_plan, solo_roles
)

print("no-analyst plan -> status:", solo_run.status,
      "| termination:", solo_run.termination_reason)
print("task records    :", [
    (r.task_id, r.role, r.status) for r in solo_run.tasks
])
print("finding fabricated:", solo_run.finding is not None, "(must be False)")


In [ ]:
# -- 3: retryable task failure (scripted test doubles) -------------------
import importlib.util

spec = importlib.util.spec_from_file_location(
    "notebook_m17_doubles", str(ROOT / "tests" / "orchestration" / "m17_doubles.py")
)
doubles = importlib.util.module_from_spec(spec)
spec.loader.exec_module(doubles)

from repo_intelligence.orchestration import OrchestrationConfig

broken = doubles.report_payload()
del broken["findings"]  # invalid structured response -> retryable

retry_orch, _inv, _ana = doubles.build(
    inv_responses=[
        doubles.tool_calls(
            doubles.call("read_file", dict(doubles.README_REF))
        ),
        doubles.structured(broken),                    # attempt 1 fails
        doubles.tool_calls(
            doubles.call("read_file", dict(doubles.README_REF))
        ),
        doubles.structured(doubles.report_payload()),  # attempt 2 commits
    ],
    config=OrchestrationConfig(max_retries=1),
    trace_sink=rq.JsonFileTraceSink(WORK / "double-traces"),
)
retry_plan = doubles.two_step_plan()
retry_roles = assign_roles(retry_plan.steps)
retry_run = lg.run_with_langgraph(
    retry_orch, doubles.task(), retry_plan, retry_roles
)

first = retry_run.tasks[0]
print("status        :", retry_run.status,
      "| retries:", retry_run.retries)
print("P1 attempts   :", first.attempts,
      "| recent tool calls:", first.usage.tool_calls,
      "| cumulative:", retry_run.usage_total.tool_calls)
print("P2 status     :", retry_run.tasks[1].status)


## 8 — Legacy versus LangGraph

The **same public interface** selects the backend:
`run_repository_query(..., orchestration="legacy" | "langgraph")` and
the CLI flag `--orchestration`. Contracts that stay equivalent: the
planner + H1 preflight, the specialists and their evidence discipline,
the role vocabulary, budgets, M16 validation, six-rule diagram
validation, and the `RepositoryQueryResult` shape.

Parity is test-pinned in
`tests/agents/test_repository_query_langgraph.py`
(`test_legacy_and_langgraph_results_are_equivalent_offline`): every
public field is equal offline; the only intentional difference is run
identity (fresh per-step run ids per path). The graph internals have
their own 34 behavioral tests in
`tests/orchestration/test_langgraph_sequential.py`.


In [ ]:
_require_graph()

def _both_backends():
    rows = {}
    for backend in ("legacy", "langgraph"):
        tmp = Path(tempfile.mkdtemp(prefix="nb_parity_"))
        rows[backend] = rq.run_repository_query(
            QUESTION,
            mode="offline",
            artifacts=False,
            db_path=str(tmp / "memory.db"),
            trace_root=str(tmp / "traces"),
            orchestration=backend,
        )
    return rows

rows = _both_backends()
legacy_result = rows["legacy"]
graph_result = rows["langgraph"]

for field in (
    "status", "termination_reason", "collaboration_status",
    "diagram_status", "answer", "conclusions", "evidence",
):
    left, right = getattr(legacy_result, field), getattr(graph_result, field)
    print("%-22s %s" % (field, "EQUAL" if left == right else "DIFFER"))
print("%-22s %s" % (
    "step shape",
    "EQUAL" if [
        (s.step_id, s.role, s.status, s.label) for s in legacy_result.steps
    ] == [
        (s.step_id, s.role, s.status, s.label) for s in graph_result.steps
    ] else "DIFFER",
))
print("run ids disjoint     :",
      set(legacy_result.run_ids).isdisjoint(set(graph_result.run_ids)))


## 9 — Safeguards (H1–H4) on the graph path

The safeguards live **below** both orchestration backends, so the
opt-in graph path inherits them unchanged:

- **H1/H2** — the planner request is budget-gated *before* the provider
  is called (`_build_plan` → `enforce_request_budget_or_raise`); every
  specialist model call passes `Agent._gate_model_request`
  (`agent/agent.py`) which clips or rejects over-budget requests
  before any provider call;
- **H3** — graph observations are projected to the compact
  `model_graph_compact.v1` payload in `live_bindings`
  (`tools/graph_projection.py`);
- **H4** — the request-budget lifecycle events are recorded per model
  call, on both backends.

Regression suites (run them, don't re-implement them here):

```bash
venv312/bin/python -m pytest tests/agent/test_request_budget.py \
    tests/agent/test_context_budget_lifecycle.py \
    tests/tools/test_graph_projection.py \
    tests/tools/test_response_ceiling_boundaries.py -q   # 70 passed
```

The demo below proves the entry-level property directly: an
oversized planner request is rejected **before the graph is even
selected** — zero provider calls, typed `PLAN_FAILED`.


In [ ]:
huge_result = rq.run_repository_query(
    "x" * 400_000,                       # ~100k estimated tokens
    mode="offline",
    artifacts=False,
    orchestration="langgraph",
)

print("status      :", huge_result.status)
print("termination :", huge_result.termination_reason)
print("first failure line:")
print("  ", huge_result.failures[0][:96])
assert huge_result.termination_reason == "PLAN_FAILED"
assert huge_result.failures[0].startswith("RequestBudgetExceededError:")
print()
print("=> the provider was never called; the graph was never selected.")


## 10 — Learning summary and next steps

**What this notebook demonstrated**

- *Stateful graph workflows*: one compiled `StateGraph` over a typed
  `LangGraphOrchestrationState`; the plan is **data** — the same
  topology executes 1-, 2-, or 3-step plans.
- *Node and edge responsibilities*: `initialize` (validate + wire),
  `select_task` (cursor + gates + analyst handoff), `execute_task`
  (the single executor dispatch), `commit_result` (honest commit or
  typed failure + retry decision), `retry_task` (attempt+1, same
  identity), `assemble_and_validate` (merge + M16 verdict).
- *Routing and conditional transitions*: explicit router keys in
  state, hard edges for the success path, conditional edges for
  select/assemble/retry/end — with budget stops that never consume a
  retry.
- *Coordinating agent roles*: INVESTIGATOR steps produce typed
  `InvestigationReport`s; the ANALYST step consumes the *merged*
  report via `dependency_results` and produces the typed
  `ArchitectureFinding`; roles are preassigned with the same
  deterministic assigner both backends share.
- *Evidence handoffs*: findings carry `evidence_refs` that survive
  task commits, the mid-run merge, final assembly, and the
  `RepositoryQueryResult` (with per-run trace paths on disk).
- *Error handling and termination*: invalid plans → `PLAN_FAILED`
  before execution; task failures → typed categories, bounded retry,
  blocked successors; no-analyst plans → `INCOMPLETE / NO_ANALYSIS`;
  M16 rejection → `VALIDATION_FAILED` — never a fabricated success.
- *Testing multi-agent systems*: scripted doubles at the LLM and
  executor seams, behavioral (not node-name) assertions, a
  never-`Orchestrator.execute` guard, and entry-point routing + parity
  tests.

**Deliberate limitations (this milestone)**

- sequential investigator execution only (no parallelism);
- planner and role assignment stay *off-graph* (composition-root
  concerns);
- no checkpointing/resume, no distributed execution, no replanning;
- the legacy orchestrator remains the default and is not removed;
- live (`mode="live"`) graph runs are wired but not exercised in this
  notebook (no credentials in the environment).

**Suggested next milestone**: promote the graph path behind a
validation-period default flip (production `orchestration` default →
`langgraph`) once live-mode parity is observed, and add
cross-step streaming of task events for observability.

**Where to go next in the source**

- `docs/langgraph-end-to-end.md` — entry-point selection, topology,
  failure/retry, safeguards, test commands;
- `docs/m21-langgraph-orchestration.md` — the graph-node contract
  tables;
- `tests/orchestration/test_langgraph_sequential.py` — the 34
  behavioral tests behind every claim above.
